In [1]:
# Import library
import torch
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from pydub import AudioSegment
import random

In [ ]:
# Load data
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

audio_file = 'sample_audio.wav'
audio, sr = librosa.load(audio_file, sr=None)
clip = AudioSegment.from_wav(audio_file)

In [ ]:
# Export Helper
def export_pydub(audio):
    while True:
        file_name = input("Input Filename: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("file must be end with .wav")
    audio.export(file_name, format="wav")
    print("Audio name save as", file_name)
    
def export_librosa(audio):
    while True:
        file_name = input("Input Filename: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("file must be end with .wav")
    sf.write(file_name, np.clip(audio, -1,1), sr)
    print("Audio name save as", file_name)

In [6]:
# Predict Text

def predict_text(audio_file):
    waveform, sr = torchaudio.load(audio_file)
    waveform = waveform.mean(dim=0, keepdim=True)
    
    if sr != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sr, bundle.sample_rate)
        
    with torch.inference_mode():
        emission, _ = model(waveform)
        
    class GreedyCTCDecoder(torch.nn.Module):
        def __init__ (self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0
            
        def forward(self, emission: torch.Tensor) -> str:
            indices = torch.argmax(emission, dim=-1)
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            return "".join([self.labels[i] for i in indices])
        
    decoder = GreedyCTCDecoder(bundle.get_labels())
    transcript = decoder(emission[0])
    transcript = transcript.replace("|", " ")
    return transcript

In [11]:
# Augmentation only
# Pitch Shift, White Noise, Repeat Clip, Insert Fade In

def pitch_shift():
    while True:
        pitch_rate = float(input("Input pitch rate (-5 to 5): "))
        if -5 <= pitch_rate <= 5:
            print("Must be between -5 to 5")
        else:
            break
    shifted_audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=pitch_rate)
    export_librosa(shifted_audio)
    
def white_noise():
    while True:
        level = float(input("Input noise level (0.1 to 0.5): "))
        if 0.1 <= level <= 0.5:
            print("Must be between 0.1 to 0.5")
        else:
            break
    noise = np.random.normal(0, audio.std(), audio.size)
    export_librosa(audio, noise * level)
    
def repeat(clip):
    while True:
        repeat = int(input("Input Repeat Clip (1-50): "))
        if 1 <= repeat <= 50:
            print("must be between 1 to 50")
        else:
            break
    export_pydub(clip * repeat)
    
def fade_in(clip):
    while True:
        duration = int(input("Input fadein duration ms (1s = 1000ms): "))
        if duration > 0:
            break
        else:
            print("must be more than 0")
    export_pydub(clip.fade_in(duration))

In [14]:
def aug_menu():
    while True:
        print("1. Pitch Shift")
        print("2. White Noise")
        print("3. Repeat Audio")
        print("4. Fade In")
        print("5. Back to Main Menu")
        
        choice = input(">> ")
        if choice == "1":
            pitch_shift()
        elif choice == "2":
            white_noise()
        elif choice == "3":
            repeat(clip)
        elif choice == "4":
            fade_in(clip)
        elif choice == "5":
            break
        else:
            print("Invalid Input must be (1 to 10)")


In [18]:
def mainmenu():
    while True:
        print("1. Audio Augmentation")
        print("2. Predict Text")
        print("3. Exit")
        
        choice = input(">> ")
        if choice == "1":
            print("Audio Augmentation Menu")
            aug_menu()
        elif choice == "2":
            transcript = predict_text(audio_file)
            print("Hasil Transkrip: ")
            print(transcript)
        elif choice == "3":
            break
        else:
            print("Invalid Input")
    print("Main Menu Closed")

mainmenu()

1. Audio Augmentation
2. Predict Text
3. Exit
Hasil Transkrip: 
THE PUBLIC ARE ENTREATED TO BEAR IN MIND THAT THIRTEEN YEARS HAVE PASSED SINCE IT WAS FINISHED MANY MORE SINCE IT WAS BEGUN AND THAT DURING THAT PERIOD PLACES MANNERS BOOKS AND OPINIONS HAVE UNDERGONE CONSIDERABLE CHANGES 
1. Audio Augmentation
2. Predict Text
3. Exit
Main Menu Closed
